# otel-inference on a free Kaggle T4

Runs vLLM (with prefix caching + native OTel traces), the pynvml GPU exporter, the
otel-inference convention collector, and the causal load generator — all inside one
Kaggle notebook — exporting to **SigNoz Cloud**.

**Before running:** Settings -> Accelerator -> **GPU T4 x2** (or x1), and add two
Kaggle Secrets: `SIGNOZ_CLOUD_ENDPOINT` (e.g. `ingest.us.signoz.cloud:443`) and
`SIGNOZ_INGESTION_KEY`.

This is the **Day-1 kill-probe** notebook: cells 2 and 4 confirm the two riskiest
assumptions (vLLM exposes the metrics; Kaggle can export OTLP to SigNoz) before you
invest in the rest.

## 0. Install

In [ ]:
!pip -q install vllm httpx nvidia-ml-py \
  opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc
# otelcol-contrib binary (transform + prometheus receiver live in contrib)
!curl -sSL -o /tmp/otelcol.tar.gz \
  https://github.com/open-telemetry/opentelemetry-collector-releases/releases/download/v0.111.0/otelcol-contrib_0.111.0_linux_amd64.tar.gz
!mkdir -p /opt/otelcol && tar -xzf /tmp/otelcol.tar.gz -C /opt/otelcol
!/opt/otelcol/otelcol-contrib --version

## 1. Pull the repo (config, collector rules, loadgen, gpu exporter)

In [ ]:
!git clone --depth 1 -b claude/signoz-hackathon-research-np9zyu \
  https://github.com/Eienel/sii.git /kaggle/working/sii
%cd /kaggle/working/sii/otel-inference

## 2. KILL-PROBE A — serve vLLM and confirm the metrics exist
Starts vLLM in the background, waits for readiness, then greps `/metrics` for the
exact names the OTTL rules depend on. If any are missing, fix `collector/ottl/vllm.yaml`
before going further.

In [ ]:
import os, subprocess, time, urllib.request
os.makedirs('/kaggle/working/logs', exist_ok=True)
vllm = subprocess.Popen(
    ['vllm','serve','Qwen/Qwen2.5-0.5B-Instruct',
     '--enable-prefix-caching','--max-model-len','4096',
     '--otlp-traces-endpoint','http://localhost:4317'],
    stdout=open('/kaggle/working/logs/vllm.log','w'), stderr=subprocess.STDOUT)
# wait for readiness
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2); print('vLLM ready'); break
    except Exception: time.sleep(5)
else:
    print('vLLM did not become ready — check logs/vllm.log')

m = urllib.request.urlopen('http://localhost:8000/metrics', timeout=5).read().decode()
need = ['vllm:prefix_cache_hits','vllm:prefix_cache_queries','vllm:kv_cache_usage_perc',
        'vllm:num_requests_running','vllm:num_requests_waiting',
        'vllm:request_queue_time_seconds','vllm:request_prefill_time_seconds',
        'vllm:request_decode_time_seconds','vllm:time_to_first_token_seconds',
        'vllm:e2e_request_latency_seconds']
for n in need:
    print(('FOUND ' if n in m else 'MISSING ')+n)

## 3. GPU exporter (pynvml -> OTLP)

In [ ]:
gpu = subprocess.Popen(['python','gpu_exporter/gpu_otel.py','--endpoint','http://localhost:4317','--interval','2'],
    stdout=open('/kaggle/working/logs/gpu.log','w'), stderr=subprocess.STDOUT)
time.sleep(3); print('gpu exporter started')

## 4. KILL-PROBE B — run the convention collector -> SigNoz Cloud
Uses Kaggle Secrets for the endpoint + key. The collector's `otlp/signoz_local`
exporter will error harmlessly if you have no local SigNoz; the cloud exporter is
what matters here.

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['SIGNOZ_CLOUD_ENDPOINT'] = s.get_secret('SIGNOZ_CLOUD_ENDPOINT')
os.environ['SIGNOZ_INGESTION_KEY'] = s.get_secret('SIGNOZ_INGESTION_KEY')
os.environ['VLLM_METRICS_TARGET'] = '127.0.0.1:8000'
os.environ['OLLAMA_METRICS_TARGET'] = '127.0.0.1:11434'

# pre-flight: validate config + OTTL rules before starting (catches syntax errors fast)
v = subprocess.run(['/opt/otelcol/otelcol-contrib','validate','--config','collector/otelcol-config.yaml'],
    capture_output=True, text=True)
print('validate rc=', v.returncode, (v.stderr or v.stdout)[-800:])
assert v.returncode == 0, 'collector config invalid — fix before continuing'

col = subprocess.Popen(['/opt/otelcol/otelcol-contrib','--config','collector/otelcol-config.yaml'],
    stdout=open('/kaggle/working/logs/collector.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
print(open('/kaggle/working/logs/collector.log').read()[-2000:])

## 5. Drive the causal load: WARM -> SPIKE -> RECOVER
Then open SigNoz and import `dashboards/inference_causal.json`. You should see the
SPIKE phase: hit-rate collapses, queue wait climbs, TTFT/e2e spike; RECOVER drains it.

In [ ]:
!python loadgen/generate.py --base-url http://localhost:8000 \
  --model Qwen/Qwen2.5-0.5B-Instruct --ramp --phase-seconds 90 --spike-concurrency 48

## 6. Teardown

In [ ]:
for p in (col, gpu, vllm):
    try: p.terminate()
    except Exception: pass
print('stopped')